# Space Weather: The Solar Perspective — Implementation
# 우주 기상: 태양 관점 — 구현

**논문 / Paper**: Schwenn, R. (2006), "Space Weather: The Solar Perspective", *Living Rev. Solar Phys.*, 3, 2

**날짜 / Date**: 2026-04-09

---

이 노트북은 Schwenn (2006) 리뷰에서 다룬 핵심 우주 기상 현상들을 수치적으로 구현하고 시각화한다.
태양풍 두 상태 모델, Parker spiral, CIR 형성, 발레리나 모델, 태양 플레어 분류, SEP 이벤트 프로파일,
CME 통계, ICME 도착 시간 예측, 그리고 태양-지구 우주 기상 영향 체인을 포괄한다.

This notebook numerically implements and visualizes the key space weather phenomena covered
in Schwenn (2006). It spans the two-state solar wind model, Parker spiral, CIR formation,
the ballerina model, solar flare classification, SEP event profiles, CME statistics,
ICME travel time prediction, and the Sun-to-Earth space weather impact chain.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib import cm
from mpl_toolkits.mplot3d import Axes3D
from scipy import constants
from scipy.integrate import odeint
import warnings
warnings.filterwarnings('ignore')

# Global plot style.
plt.rcParams.update({
    'figure.figsize': (10, 6),
    'font.size': 12,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'figure.dpi': 100,
})

# Physical constants from scipy.
AU = constants.au  # 1 AU in meters
R_SUN = 6.957e8  # Solar radius in meters
M_PROTON = constants.m_p  # Proton mass in kg
K_B = constants.k  # Boltzmann constant
OMEGA_SUN = 2 * np.pi / (25.4 * 86400)  # Solar angular velocity (sidereal, rad/s)

print("All imports successful. / 모든 임포트 완료.")
print(f"1 AU = {AU:.3e} m")
print(f"Solar rotation period (sidereal) = 25.4 days")
print(f"Omega_sun = {OMEGA_SUN:.3e} rad/s")

---
## Part 1: 태양풍 두 상태 모델 / Solar Wind Two-State Model

Schwenn (2006) Table 1은 태양풍의 두 가지 기본 상태를 비교한다:
- **고속 태양풍 (HSS, High-Speed Streams)**: 코로나 홀에서 기원, 속도 ~600-800 km/s
- **저속 태양풍 (LSM, Low-Speed Mode)**: 스트리머 벨트에서 기원, 속도 ~300-400 km/s

놀라운 점은 운동량 플럭스(momentum flux)와 에너지 플럭스(energy flux) 밀도가
두 상태에서 거의 동일하다는 것이다. 이는 태양풍 가속 메커니즘에 대한 근본적 제약 조건을 제공한다.

Schwenn (2006) Table 1 compares the two fundamental states of the solar wind:
- **High-Speed Streams (HSS)**: originate from coronal holes, speed ~600-800 km/s
- **Low-Speed Mode (LSM)**: originate from the streamer belt, speed ~300-400 km/s

Remarkably, the momentum flux density and energy flux density are nearly
identical between the two states. This provides a fundamental constraint
on the solar wind acceleration mechanism.

In [ ]:
# === Solar Wind Two-State Model (Table 1, Schwenn 2006) ===

# Parameters from Table 1 (at 1 AU).
params = {
    'Speed (km/s)':              {'HSS': 700,   'LSM': 350},
    'Density (cm^-3)':           {'HSS': 3,     'LSM': 10},
    'Proton flux\n(10^8 cm^-2 s^-1)': {'HSS': 2,  'LSM': 3.5},
    'Temperature\n(10^3 K)':     {'HSS': 230,   'LSM': 50},
    'Mom. flux\n(nPa)':         {'HSS': 2.4,   'LSM': 2.1},
    'Energy flux\n(erg cm^-2 s^-1)': {'HSS': 1.5, 'LSM': 0.6},
    'He content\n(%)':          {'HSS': 5,     'LSM': 2},
}

# Bar chart comparison.
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()
colors = {'HSS': '#D32F2F', 'LSM': '#1976D2'}

for idx, (param_name, values) in enumerate(params.items()):
    ax = axes[idx]
    bars = ax.bar(
        list(values.keys()),
        list(values.values()),
        color=[colors[k] for k in values.keys()],
        edgecolor='black', linewidth=0.8, width=0.5
    )
    ax.set_title(param_name, fontsize=11)
    for bar, val in zip(bars, values.values()):
        ax.text(
            bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.02,
            f'{val}', ha='center', va='bottom', fontsize=10, fontweight='bold'
        )
    ax.set_ylim(0, max(values.values()) * 1.3)
    ax.grid(axis='y', alpha=0.3)
    ax.grid(axis='x', visible=False)

# Highlight momentum flux invariance in the 5th panel.
axes[4].axhline(y=2.25, color='green', linestyle='--', linewidth=2, alpha=0.7)
axes[4].annotate(
    'Approx. invariant!', xy=(0.5, 2.3), fontsize=10,
    color='green', fontweight='bold', ha='center'
)

# Remove unused subplot.
axes[7].axis('off')

fig.suptitle(
    'Solar Wind Two-State Model - Table 1 (Schwenn 2006)\n'
    'Solar wind two-state model - Table 1',
    fontsize=14, fontweight='bold'
)
plt.tight_layout()
plt.show()

In [ ]:
# === Ram Pressure Calculation: P_ram = 1/2 * rho * v^2 ===

# HSS parameters.
v_hss = 700e3  # m/s
n_hss = 3e6  # m^-3 (3 cm^-3)
rho_hss = n_hss * M_PROTON  # kg/m^3
P_ram_hss = 0.5 * rho_hss * v_hss**2  # Pa

# LSM parameters.
v_lsm = 350e3  # m/s
n_lsm = 10e6  # m^-3 (10 cm^-3)
rho_lsm = n_lsm * M_PROTON  # kg/m^3
P_ram_lsm = 0.5 * rho_lsm * v_lsm**2  # Pa

print("=" * 50)
print("Ram Pressure P_ram = 1/2 * rho * v^2")
print("=" * 50)
print(f"\nHSS (High-Speed Stream):")
print(f"  v = {v_hss/1e3:.0f} km/s, n = {n_hss/1e6:.0f} cm^-3")
print(f"  rho = {rho_hss:.3e} kg/m^3")
print(f"  P_ram = {P_ram_hss:.3e} Pa = {P_ram_hss*1e9:.2f} nPa")

print(f"\nLSM (Low-Speed Mode):")
print(f"  v = {v_lsm/1e3:.0f} km/s, n = {n_lsm/1e6:.0f} cm^-3")
print(f"  rho = {rho_lsm:.3e} kg/m^3")
print(f"  P_ram = {P_ram_lsm:.3e} Pa = {P_ram_lsm*1e9:.2f} nPa")

print(f"\nRatio P_ram(HSS)/P_ram(LSM) = {P_ram_hss/P_ram_lsm:.2f}")
print(f"\n-> Both states have similar ram pressure (~2 nPa),")
print(f"   consistent with the momentum flux invariance.")
print(f"-> 두 상태의 ram pressure가 비슷한 크기 (~2 nPa)를 가진다.")
print(f"   이는 momentum flux 불변성과 일치한다.")

---
## Part 2: Parker Spiral과 CIR 형성 / Parker Spiral and CIR Formation

Parker spiral은 태양풍 입자가 방사 방향으로 직선 이동하면서,
자기장선이 태양 자전에 의해 뒤로 감기는 형태를 기술한다:

$$r - r_0 = -\frac{V}{\Omega_\odot}(\phi - \phi_0)$$

느린 바람(350 km/s)은 더 단단히 감기고, 빠른 바람(700 km/s)은 느슨하게 감긴다.
CIR(Corotating Interaction Region)은 빠른 바람이 앞서가는 느린 바람을 따라잡을 때 형성된다.

The Parker spiral describes how magnetic field lines are wound backward by solar
rotation while solar wind particles travel radially outward:

$$r - r_0 = -\frac{V}{\Omega_\odot}(\phi - \phi_0)$$

Slow wind (350 km/s) is wound more tightly, while fast wind (700 km/s) is more
loosely wound. CIRs form when fast wind overtakes the preceding slow wind.

In [ ]:
# === Parker Spiral Field Lines ===

def parker_spiral(phi, v_sw, r0=R_SUN):
    """Compute Parker spiral radial distance for given azimuthal angle.

    Args:
        phi: Azimuthal angle (radians) relative to source.
        v_sw: Solar wind speed (m/s).
        r0: Source surface radius (m).

    Returns:
        Radial distance (m).
    """
    return r0 - (v_sw / OMEGA_SUN) * phi


fig, ax = plt.subplots(1, 1, figsize=(10, 10), subplot_kw={'projection': 'polar'})

speeds = [
    (700e3, '#D32F2F', 'Fast wind (700 km/s)'),
    (350e3, '#1976D2', 'Slow wind (350 km/s)'),
]

# Plot multiple field lines from different source longitudes.
for v_sw, color, label in speeds:
    for phi0 in np.linspace(0, 2 * np.pi, 5, endpoint=False):
        # Parametric form: for r from r0 to 1.5 AU.
        r_range = np.linspace(R_SUN, 1.5 * AU, 1000)
        phi_vals = phi0 - OMEGA_SUN / v_sw * (r_range - R_SUN)
        r_au = r_range / AU

        lbl = label if phi0 == 0 else None
        ax.plot(phi_vals, r_au, color=color, linewidth=1.5, label=lbl, alpha=0.8)

# Mark Earth orbit.
theta_earth = np.linspace(0, 2 * np.pi, 200)
ax.plot(theta_earth, np.ones_like(theta_earth), 'k--', linewidth=1, alpha=0.4, label='1 AU')

# Sun at center.
ax.plot(0, 0, 'yo', markersize=15, zorder=5)

ax.set_rlim(0, 1.5)
ax.set_title(
    'Parker Spiral Field Lines\nParker spiral magnetic field lines',
    fontsize=14, fontweight='bold', pad=20
)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0), fontsize=10)
plt.tight_layout()
plt.show()

# Compute garden-hose angle at 1 AU.
for v_sw, name in [(700e3, 'Fast'), (350e3, 'Slow')]:
    angle = np.degrees(np.arctan(OMEGA_SUN * AU / v_sw))
    print(f"{name} wind: Garden-hose angle at 1 AU = {angle:.1f} deg")

In [ ]:
# === CIR Formation: Speed Profile Evolution ===

def cir_speed_profile(phi_deg, r_au, v_fast=700, v_slow=350, width_deg=90):
    """Simulate solar wind speed profile evolution with distance.

    At the Sun, the speed profile is a rectangular step function.
    With increasing distance, the fast-slow interface steepens on the
    leading edge (compression) and rarefies on the trailing edge.

    Args:
        phi_deg: Longitude array (degrees).
        r_au: Heliocentric distance (AU).
        v_fast: Fast wind speed (km/s).
        v_slow: Slow wind speed (km/s).
        width_deg: Width of fast stream at the source (degrees).

    Returns:
        Speed profile at given distance (km/s).
    """
    # Travel time difference causes the fast wind to "catch up".
    delta_phi = OMEGA_SUN * r_au * AU * (1 / (v_slow * 1e3) - 1 / (v_fast * 1e3))
    delta_phi_deg = np.degrees(delta_phi)

    # Smoothing increases with distance to simulate CIR formation.
    smoothing = 5 + 20 * r_au

    # Rectangular profile at source, distorted with distance.
    profile = np.zeros_like(phi_deg, dtype=float)
    for p_idx, p in enumerate(phi_deg):
        # Shift leading edge by differential rotation effect.
        leading_edge = 180 - width_deg / 2 - delta_phi_deg * 0.5
        trailing_edge = 180 + width_deg / 2 + delta_phi_deg * 0.3
        # Sigmoid transitions with distance-dependent steepness.
        rise = 1 / (1 + np.exp(-(p - leading_edge) / (smoothing * 0.3)))
        fall = 1 / (1 + np.exp(-(p - trailing_edge) / smoothing))
        profile[p_idx] = v_slow + (v_fast - v_slow) * (rise - fall)

    return np.clip(profile, v_slow, v_fast)


fig, axes = plt.subplots(2, 2, figsize=(14, 10))
phi_deg = np.linspace(0, 360, 500)

distances = [0.0, 0.3, 0.7, 1.0]
titles = [
    'At Sun (r ~ 0) / 태양 표면',
    'r = 0.3 AU',
    'r = 0.7 AU',
    'r = 1.0 AU (Earth) / 지구 궤도'
]

for idx, (r, title) in enumerate(zip(distances, titles)):
    ax = axes.flatten()[idx]
    speed = cir_speed_profile(phi_deg, r)
    ax.fill_between(phi_deg, 300, speed, alpha=0.3, color='#D32F2F',
                    where=speed > 400)
    ax.fill_between(phi_deg, 300, speed, alpha=0.3, color='#1976D2',
                    where=speed <= 400)
    ax.plot(phi_deg, speed, 'k-', linewidth=2)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Carrington Longitude (deg)')
    ax.set_ylabel('Speed (km/s)')
    ax.set_ylim(280, 770)
    ax.set_xlim(0, 360)

    if r > 0:
        # Mark CIR compression region.
        grad = np.gradient(speed, phi_deg)
        steep_idx = np.argmax(np.abs(grad))
        ax.axvline(phi_deg[steep_idx], color='green', linestyle='--',
                   alpha=0.7, label='Stream interface')
        ax.legend(fontsize=9)

fig.suptitle(
    'CIR Formation: Speed Profile Evolution with Distance\n'
    'CIR 형성: 거리에 따른 속도 프로파일 진화 (Figure 10 concept)',
    fontsize=14, fontweight='bold'
)
plt.tight_layout()
plt.show()

---
## Part 3: 발레리나 모델과 3D 태양권 / Ballerina Model and 3D Heliosphere

Alfven (1977)의 발레리나 모델은 태양권 전류면(Heliospheric Current Sheet, HCS)을
회전하는 발레리나의 스커트에 비유한다. HCS는 태양 자기 쌍극자의 기울기(tilt angle)에 따라
뒤틀리며, 이 뒤틀림은 태양 활동 주기에 따라 변한다.

- 태양 활동 극소기: tilt angle이 작아 HCS가 거의 평평함
- 태양 활동 극대기: tilt angle이 크고, 다극자 성분이 강해져 복잡한 구조

Alfven's (1977) ballerina model likens the Heliospheric Current Sheet (HCS)
to a spinning ballerina's skirt. The HCS is warped according to the tilt angle
of the solar magnetic dipole, which varies with the solar activity cycle.

- Solar minimum: small tilt angle, nearly flat HCS
- Solar maximum: large tilt angle, complex multipolar structure

In [ ]:
# === Warped Heliospheric Current Sheet (Ballerina Skirt) ===

def heliospheric_current_sheet(r_au, phi, tilt_deg, n_warp=1):
    """Compute the latitude of the heliospheric current sheet.

    The HCS latitude is determined by the tilt of the solar dipole.
    A quadrupole component adds additional warping.

    Args:
        r_au: Radial distance (AU), used for Parker spiral winding.
        phi: Azimuthal angle (radians).
        tilt_deg: Dipole tilt angle (degrees).
        n_warp: Quadrupole warping order.

    Returns:
        HCS latitude (radians).
    """
    tilt = np.radians(tilt_deg)
    # Dipole contribution.
    theta_hcs = tilt * np.sin(phi)
    # Small quadrupole contribution.
    theta_hcs += 0.15 * tilt * np.sin(2 * phi)
    return theta_hcs


fig = plt.figure(figsize=(18, 6))
tilt_angles = [10, 30, 60]
subtitles = [
    'Solar minimum\n태양 활동 극소기\n(tilt = 10 deg)',
    'Intermediate\n중간 단계\n(tilt = 30 deg)',
    'Solar maximum\n태양 활동 극대기\n(tilt = 60 deg)'
]

for idx, (tilt, subtitle) in enumerate(zip(tilt_angles, subtitles)):
    ax = fig.add_subplot(1, 3, idx + 1, projection='3d')

    # Create the warped sheet.
    r_vals = np.linspace(0.1, 1.5, 80)
    phi_vals = np.linspace(0, 4 * np.pi, 200)  # Two full rotations.
    R, PHI = np.meshgrid(r_vals, phi_vals)

    # Account for Parker spiral winding.
    phi_shifted = PHI + OMEGA_SUN * R * AU / (400e3)  # ~400 km/s average.
    THETA = heliospheric_current_sheet(R, phi_shifted, tilt)

    # Convert to Cartesian.
    X = R * np.cos(PHI)
    Y = R * np.sin(PHI)
    Z = R * np.sin(THETA) * 0.3  # Scale z for visibility.

    ax.plot_surface(
        X, Y, Z, alpha=0.6,
        cmap='coolwarm', edgecolor='none',
        rstride=2, cstride=2
    )

    # Sun at center.
    ax.scatter([0], [0], [0], color='yellow', s=200, edgecolor='orange',
               linewidth=2, zorder=5)

    ax.set_xlim(-1.5, 1.5)
    ax.set_ylim(-1.5, 1.5)
    ax.set_zlim(-0.5, 0.5)
    ax.set_xlabel('X (AU)', fontsize=8)
    ax.set_ylabel('Y (AU)', fontsize=8)
    ax.set_zlabel('Z (AU)', fontsize=8)
    ax.set_title(subtitle, fontsize=10, fontweight='bold')
    ax.view_init(elev=25, azim=45)

fig.suptitle(
    'Ballerina Model: Warped Heliospheric Current Sheet\n'
    '발레리나 모델: 뒤틀린 태양권 전류면',
    fontsize=14, fontweight='bold'
)
plt.tight_layout()
plt.show()

In [ ]:
# === Bz Variations at Sector Crossings (Rosenberg & Coleman Mechanism) ===

# At sector boundaries, the HCS crosses the ecliptic plane, and the
# magnetic field direction changes sign. The out-of-ecliptic component Bz
# has systematic variations near sector boundaries.

time_days = np.linspace(0, 27, 1000)  # One Carrington rotation.
phi_time = 2 * np.pi * time_days / 27.0

# Simulate HCS crossings for tilt = 20 deg.
tilt = 20
hcs_lat = heliospheric_current_sheet(1.0, phi_time, tilt)

# Polarity: above HCS -> away (positive), below -> toward (negative).
# Observer at ecliptic (latitude = 0).
polarity = np.sign(-hcs_lat)  # Simplified.

# Bz variations near sector boundaries (Rosenberg-Coleman effect).
B_total = 5.0  # nT, typical IMF magnitude at 1 AU.
# Parker spiral angle at 1 AU.
psi = np.arctan(OMEGA_SUN * AU / (400e3))  # ~45 deg for 400 km/s.
Br = B_total * np.cos(psi) * polarity
Bt = -B_total * np.sin(psi) * polarity

# Bz arises from the out-of-ecliptic draping near HCS.
Bz = -B_total * 0.3 * np.sin(2 * phi_time) * np.exp(-np.abs(hcs_lat) / 0.2)

fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)

axes[0].fill_between(time_days, 0, polarity, alpha=0.4,
                     where=polarity > 0, color='red', label='Away (+)')
axes[0].fill_between(time_days, 0, polarity, alpha=0.4,
                     where=polarity < 0, color='blue', label='Toward (-)')
axes[0].set_ylabel('Polarity')
axes[0].set_title('Magnetic Sector Structure / 자기 섹터 구조')
axes[0].legend(loc='upper right')
axes[0].set_ylim(-1.5, 1.5)

axes[1].plot(time_days, Br, 'r-', label='$B_r$', linewidth=1.5)
axes[1].plot(time_days, Bt, 'b-', label='$B_t$', linewidth=1.5)
axes[1].set_ylabel('B (nT)')
axes[1].set_title('IMF Components in Ecliptic / 황도면 IMF 성분')
axes[1].legend()

axes[2].plot(time_days, Bz, 'g-', linewidth=2)
axes[2].fill_between(time_days, 0, Bz, where=Bz < 0, alpha=0.3, color='green')
axes[2].axhline(0, color='k', linewidth=0.5)
axes[2].set_ylabel('$B_z$ (nT)')
axes[2].set_xlabel('Time (days) / 시간 (일)')
axes[2].set_title(
    '$B_z$ Variations at Sector Crossings (Rosenberg-Coleman) / '
    '섹터 경계에서의 $B_z$ 변동'
)

fig.suptitle(
    'Sector Structure and $B_z$ at HCS Crossings\n'
    '섹터 구조와 HCS 횡단 시 $B_z$ 변동',
    fontsize=14, fontweight='bold'
)
plt.tight_layout()
plt.show()

---
## Part 4: 태양 플레어 분류와 라디오 버스트 / Solar Flare Classification and Radio Bursts

### GOES X-ray 분류 / GOES X-ray Classification
태양 플레어는 GOES 위성의 1-8 A 연X선(soft X-ray) 피크 플럭스로 분류한다:
A, B, C, M, X -- 각 등급은 10배 차이.

Solar flares are classified by their peak 1-8 A soft X-ray flux measured by GOES:
A, B, C, M, X -- each class is a factor of 10.

### 라디오 버스트 / Radio Bursts
- **Type III**: 전자 빔이 코로나를 따라 외향 전파. $f_p = 9\sqrt{n_e}$ kHz
- **Type II**: CME 구동 충격파에서 방출. 느린 주파수 드리프트.

- **Type III**: Electron beams propagating outward through corona. $f_p = 9\sqrt{n_e}$ kHz
- **Type II**: Emission from CME-driven shocks. Slow frequency drift.

In [ ]:
# === GOES X-ray Flare Classification System ===

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Panel 1: Classification scale.
classes = ['A', 'B', 'C', 'M', 'X']
flux_ranges = [
    (1e-8, 1e-7),  # A
    (1e-7, 1e-6),  # B
    (1e-6, 1e-5),  # C
    (1e-5, 1e-4),  # M
    (1e-4, 1e-3),  # X
]
colors_flare = ['#4CAF50', '#8BC34A', '#FFC107', '#FF9800', '#F44336']

for i, (cls, (f_low, f_high), color) in enumerate(
    zip(classes, flux_ranges, colors_flare)
):
    ax1.barh(
        i, np.log10(f_high) - np.log10(f_low),
        left=np.log10(f_low), height=0.7,
        color=color, edgecolor='black', linewidth=1
    )
    ax1.text(
        (np.log10(f_low) + np.log10(f_high)) / 2, i,
        f'{cls}\n{f_low:.0e} to {f_high:.0e}',
        ha='center', va='center', fontsize=10, fontweight='bold'
    )

ax1.set_yticks(range(len(classes)))
ax1.set_yticklabels(classes)
ax1.set_xlabel('log10(Peak Flux) [W/m^2]')
ax1.set_title(
    'GOES X-ray Flare Classification\nGOES X선 플레어 분류',
    fontweight='bold'
)

# Panel 2: Simulated GOES light curve for an X-class flare.
t = np.linspace(-10, 60, 1000)  # Minutes.


def goes_lightcurve(t, t_peak=5, tau_rise=3, tau_decay=15, peak_flux=5e-4):
    """Generate a synthetic GOES soft X-ray light curve.

    Args:
        t: Time array (minutes).
        t_peak: Time of peak (minutes).
        tau_rise: Rise time constant (minutes).
        tau_decay: Decay time constant (minutes).
        peak_flux: Peak flux (W/m^2).

    Returns:
        Flux array (W/m^2).
    """
    flux = np.where(
        t < t_peak,
        peak_flux * np.exp(-((t - t_peak) / tau_rise) ** 2),
        peak_flux * np.exp(-(t - t_peak) / tau_decay)
    )
    # Add background.
    flux += 1e-7
    return flux


flux = goes_lightcurve(t)
ax2.semilogy(t, flux, 'r-', linewidth=2, label='1-8 A')
ax2.semilogy(t, flux * 0.08, 'b-', linewidth=1.5, alpha=0.7, label='0.5-4 A')

# Classification thresholds.
for cls, (_, f_high), color in zip(classes, flux_ranges, colors_flare):
    ax2.axhline(f_high, color=color, linestyle='--', alpha=0.5)
    ax2.text(55, f_high * 1.2, cls, fontsize=10, fontweight='bold', color=color)

ax2.set_xlabel('Time (minutes) / 시간 (분)')
ax2.set_ylabel('Flux (W/m^2)')
ax2.set_ylim(1e-8, 1e-3)
ax2.set_title(
    'Synthetic X5.0 Flare Light Curve\n합성 X5.0 플레어 광도 곡선',
    fontweight='bold'
)
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
# === Type III and Type II Radio Burst Simulation ===

def coronal_density_model(r_rsun):
    """Newkirk coronal electron density model.

    Args:
        r_rsun: Heliocentric distance in solar radii.

    Returns:
        Electron density in cm^-3.
    """
    # Newkirk (1961) model: n_e = n0 * 10^(4.32/r).
    n0 = 4.2e4  # cm^-3
    return n0 * 10 ** (4.32 / r_rsun)


def plasma_frequency(n_e):
    """Compute plasma frequency from electron density.

    f_p = 9 * sqrt(n_e) kHz, where n_e is in cm^-3.

    Args:
        n_e: Electron density (cm^-3).

    Returns:
        Plasma frequency (kHz).
    """
    return 9.0 * np.sqrt(n_e)


fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Panel 1: Plasma frequency vs heliocentric distance.
r_range = np.linspace(1.1, 215, 1000)  # 1.1 R_sun to ~1 AU.
n_e = coronal_density_model(r_range)
f_p = plasma_frequency(n_e)

ax = axes[0]
ax.loglog(r_range, f_p / 1e3, 'b-', linewidth=2)  # Convert kHz to MHz.
ax.set_xlabel('Heliocentric Distance ($R_{\\odot}$)')
ax.set_ylabel('Plasma Frequency (MHz)')
ax.set_title(
    '$f_p = 9\\sqrt{n_e}$ vs Distance\n'
    'Plasma frequency vs heliocentric distance',
    fontweight='bold'
)
ax.axhline(0.020, color='r', linestyle='--', alpha=0.5)
ax.text(150, 0.025, '~20 kHz\n(1 AU)', fontsize=9, color='r')

# Panel 2: Dynamic spectrum -- Type III burst.
ax = axes[1]
t_type3 = np.linspace(0, 30, 500)  # Minutes.
freq_type3 = np.linspace(0.02, 300, 400)  # MHz.
T3, F3 = np.meshgrid(t_type3, freq_type3)

# Type III: fast frequency drift (electron beam at ~c/3).
drift_center = 200 * np.exp(-t_type3 / 5)  # Exponential drift.
intensity = np.zeros_like(T3)
for i, t_val in enumerate(t_type3):
    fc = drift_center[i]
    intensity[:, i] = np.exp(-((np.log10(freq_type3) - np.log10(fc)) ** 2) / 0.1)

ax.pcolormesh(T3, F3, intensity, cmap='hot', shading='auto')
ax.set_yscale('log')
ax.set_xlabel('Time (minutes) / 시간 (분)')
ax.set_ylabel('Frequency (MHz)')
ax.set_title(
    'Type III Burst (Electron Beam)\n'
    'Type III 버스트 (전자 빔)',
    fontweight='bold'
)
ax.set_ylim(0.02, 300)

# Panel 3: Dynamic spectrum -- Type II burst.
ax = axes[2]
t_type2 = np.linspace(0, 120, 500)  # Minutes.
freq_type2 = np.linspace(0.1, 200, 400)  # MHz.
T2, F2 = np.meshgrid(t_type2, freq_type2)

# Type II: slow frequency drift (shock at ~1000 km/s).
drift_center_2 = 150 * np.exp(-t_type2 / 60)  # Slower drift.
intensity_2 = np.zeros_like(T2)
for i, t_val in enumerate(t_type2):
    fc = drift_center_2[i]
    # Fundamental and harmonic.
    intensity_2[:, i] = (
        np.exp(-((np.log10(freq_type2) - np.log10(fc)) ** 2) / 0.03) +
        0.6 * np.exp(-((np.log10(freq_type2) - np.log10(2 * fc)) ** 2) / 0.03)
    )

ax.pcolormesh(T2, F2, intensity_2, cmap='hot', shading='auto')
ax.set_yscale('log')
ax.set_xlabel('Time (minutes) / 시간 (분)')
ax.set_ylabel('Frequency (MHz)')
ax.set_title(
    'Type II Burst (CME Shock)\n'
    'Type II 버스트 (CME 충격파)',
    fontweight='bold'
)
ax.set_ylim(0.1, 200)
ax.text(80, 100, 'Harmonic', color='white', fontsize=10, fontstyle='italic')
ax.text(80, 20, 'Fundamental', color='white', fontsize=10, fontstyle='italic')

plt.tight_layout()
plt.show()

---
## Part 5: SEP 이벤트 프로파일 -- 임펄시브 vs 점진적 / SEP Event Profiles -- Impulsive vs Gradual

SEP 이벤트의 시간-강도 프로파일은 관측자의 자기적 연결(magnetic connection)에 따라
크게 달라진다 (Figure 23 concept):

- **서쪽 관측자 (well-connected)**: CME nose와 자기적으로 잘 연결 -> 빠른 상승 + 빠른 하강
- **중앙 관측자**: 충격파 flanks와 연결, 충격파 통과 시 피크
- **동쪽 관측자**: 초기에 충격파 뒤에 위치 -> 느린 상승, 충격파 통과 후 피크

SEP time-intensity profiles vary dramatically with the observer's magnetic
connection to the CME shock (Figure 23 concept):

- **Western observer (well-connected)**: magnetically connected to shock nose -> rapid rise + decline
- **Central observer**: connected to shock flanks, peak at shock passage
- **Eastern observer**: initially behind shock -> slow rise, peak after shock passage

In [ ]:
# === SEP Time-Intensity Profiles for Different Observer Longitudes ===

def sep_profile_western(t, onset=1, peak_t=3, tau_rise=2, tau_decay=10):
    """SEP profile for western (well-connected) observer.

    Rapid rise to peak, followed by exponential decay.

    Args:
        t: Time array (hours).
        onset: Onset time (hours).
        peak_t: Peak time (hours).
        tau_rise: Rise time constant (hours).
        tau_decay: Decay time constant (hours).

    Returns:
        Intensity profile (arbitrary units).
    """
    intensity = np.zeros_like(t)
    mask_rise = (t >= onset) & (t < peak_t)
    mask_decay = t >= peak_t
    intensity[mask_rise] = np.exp(-((t[mask_rise] - peak_t) / tau_rise) ** 2)
    intensity[mask_decay] = np.exp(-(t[mask_decay] - peak_t) / tau_decay)
    return intensity * 1e3


def sep_profile_central(t, onset=5, shock_t=40, tau_rise=15, tau_decay=8):
    """SEP profile for central observer.

    Gradual rise with plateau, peak near shock passage.

    Args:
        t: Time array (hours).
        onset: Onset time (hours).
        shock_t: Shock passage time (hours).
        tau_rise: Rise time constant (hours).
        tau_decay: Decay time constant (hours).

    Returns:
        Intensity profile (arbitrary units).
    """
    intensity = np.zeros_like(t)
    mask = t >= onset
    # Gradual rise to shock passage.
    pre_shock = mask & (t < shock_t)
    intensity[pre_shock] = 300 * (1 - np.exp(-(t[pre_shock] - onset) / tau_rise))
    # Peak at shock + decay.
    post_shock = t >= shock_t
    peak_val = 300 * (1 - np.exp(-(shock_t - onset) / tau_rise))
    intensity[post_shock] = (peak_val * 1.5 *
                             np.exp(-(t[post_shock] - shock_t) / tau_decay))
    return intensity


def sep_profile_eastern(t, onset=10, shock_t=50, tau_rise=25, tau_decay=12):
    """SEP profile for eastern (poorly-connected) observer.

    Very slow rise, peak well after shock passage.

    Args:
        t: Time array (hours).
        onset: Onset time (hours).
        shock_t: Shock passage time (hours).
        tau_rise: Rise time constant (hours).
        tau_decay: Decay time constant (hours).

    Returns:
        Intensity profile (arbitrary units).
    """
    intensity = np.zeros_like(t)
    mask = t >= onset
    # Very slow rise.
    pre_shock = mask & (t < shock_t)
    intensity[pre_shock] = 100 * ((t[pre_shock] - onset) / (shock_t - onset)) ** 2
    # Sharp jump at shock + decay.
    post_shock = t >= shock_t
    intensity[post_shock] = (200 *
                             np.exp(-(t[post_shock] - shock_t) / tau_decay))
    return intensity


fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Panel 1: Three observer positions.
t = np.linspace(0, 80, 1000)  # Hours.
ax = axes[0]

ax.semilogy(t, sep_profile_western(t) + 0.1, 'r-', linewidth=2,
            label='Western (W60) -- well-connected')
ax.semilogy(t, sep_profile_central(t) + 0.1, 'g-', linewidth=2,
            label='Central (W20) -- flank connection')
ax.semilogy(t, sep_profile_eastern(t) + 0.1, 'b-', linewidth=2,
            label='Eastern (E20) -- behind shock')

# Mark shock passage times.
for shock_t, color in [(40, 'g'), (50, 'b')]:
    ax.axvline(shock_t, color=color, linestyle=':', alpha=0.5)

ax.set_xlabel('Time after flare (hours) / 플레어 후 시간')
ax.set_ylabel('Proton Intensity (pfu)')
ax.set_title(
    'SEP Profiles for Three Observer Longitudes (Fig. 23 concept)\n'
    '세 관측자 경도에서의 SEP 프로파일',
    fontweight='bold'
)
ax.set_ylim(0.1, 2000)
ax.legend(fontsize=9)

# Panel 2: Impulsive vs Gradual characteristics comparison.
ax = axes[1]
categories = [
    'Duration\n(hours)',
    '3He/4He\n(x100)',
    'Fe/O\n(relative)',
    'Electron/\nProton',
    'Longitude\ncone (deg)',
]
impulsive_vals = [1, 100, 10, 10, 30]
gradual_vals = [24, 0.5, 1, 1, 180]

x = np.arange(len(categories))
width = 0.35
bars1 = ax.bar(x - width / 2, impulsive_vals, width, color='#FF7043',
               edgecolor='black', label='Impulsive')
bars2 = ax.bar(x + width / 2, gradual_vals, width, color='#42A5F5',
               edgecolor='black', label='Gradual')

ax.set_xticks(x)
ax.set_xticklabels(categories, fontsize=9)
ax.set_yscale('log')
ax.set_ylabel('Value (log scale)')
ax.set_title(
    'Impulsive vs Gradual SEP Events\n'
    '임펄시브 vs 점진적 SEP 이벤트 특성',
    fontweight='bold'
)
ax.legend()

plt.tight_layout()
plt.show()

---
## Part 6: CME 특성과 통계 / CME Properties and Statistics

Schwenn (2006)은 SOHO/LASCO 관측 기반으로 CME의 통계적 특성을 정리한다:
- 속도 범위: 수 km/s ~ 3000 km/s (평균 ~490 km/s)
- 질량: $10^{13}$ ~ $10^{16}$ g
- 운동 에너지: $10^{27}$ ~ $10^{33}$ erg
- $V_{rad} = 0.88 \times V_{exp}$: 방사 속도와 팽창 속도의 경험적 관계

Schwenn (2006) summarizes CME statistical properties from SOHO/LASCO observations:
- Speed range: a few km/s to ~3000 km/s (mean ~490 km/s)
- Mass: $10^{13}$ to $10^{16}$ g
- Kinetic energy: $10^{27}$ to $10^{33}$ erg
- $V_{rad} = 0.88 \times V_{exp}$: empirical relation between radial and expansion speeds

In [ ]:
# === CME Properties and Statistics ===

np.random.seed(42)

# Generate synthetic CME population based on observed distributions.
n_cmes = 1000

# Speed distribution: log-normal, centered around ~400 km/s.
cme_speeds = np.random.lognormal(mean=np.log(400), sigma=0.5, size=n_cmes)
cme_speeds = np.clip(cme_speeds, 20, 3500)

# Mass distribution: log-normal, centered around ~10^15 g.
cme_masses = 10 ** np.random.normal(loc=15, scale=0.7, size=n_cmes)  # grams
cme_masses = np.clip(cme_masses, 1e13, 1e17)

# Kinetic energy: 1/2 * m * v^2.
cme_ke = 0.5 * (cme_masses * 1e-3) * (cme_speeds * 1e3) ** 2  # Joules
cme_ke_erg = cme_ke * 1e7  # Convert to erg.

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Panel 1: Speed distribution.
ax = axes[0, 0]
ax.hist(cme_speeds, bins=50, color='#42A5F5', edgecolor='black',
        linewidth=0.5, alpha=0.8)
ax.axvline(np.median(cme_speeds), color='red', linestyle='--', linewidth=2,
           label=f'Median = {np.median(cme_speeds):.0f} km/s')
ax.set_xlabel('CME Speed (km/s)')
ax.set_ylabel('Count')
ax.set_title(
    'CME Speed Distribution / CME 속도 분포',
    fontweight='bold'
)
ax.legend()

# Panel 2: Mass vs Speed.
ax = axes[0, 1]
scatter = ax.scatter(
    cme_speeds, cme_masses, c=np.log10(cme_ke_erg),
    cmap='hot_r', s=10, alpha=0.5, edgecolors='none'
)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('CME Speed (km/s)')
ax.set_ylabel('Mass (g)')
ax.set_title(
    'Mass vs Speed / 질량 vs 속도',
    fontweight='bold'
)
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('log10(KE) [erg]')

# Panel 3: Kinetic energy distribution.
ax = axes[1, 0]
ax.hist(np.log10(cme_ke_erg), bins=50, color='#FF7043',
        edgecolor='black', linewidth=0.5, alpha=0.8)
ax.set_xlabel('log10(Kinetic Energy) [erg]')
ax.set_ylabel('Count')
ax.set_title(
    'CME Kinetic Energy Distribution / CME 운동 에너지 분포',
    fontweight='bold'
)

# Panel 4: V_rad = 0.88 * V_exp relation.
ax = axes[1, 1]
v_exp = np.linspace(100, 2500, 200)
v_rad_fit = 0.88 * v_exp

# Generate scattered data around the fit.
n_pts = 150
v_exp_data = np.random.uniform(100, 2000, n_pts)
v_rad_data = 0.88 * v_exp_data + np.random.normal(0, 80, n_pts)
v_rad_data = np.clip(v_rad_data, 50, 2500)

ax.scatter(v_exp_data, v_rad_data, s=20, alpha=0.5, color='#42A5F5',
           edgecolors='none', label='Data')
ax.plot(v_exp, v_rad_fit, 'r-', linewidth=2,
        label='$V_{rad} = 0.88 \\times V_{exp}$')
ax.plot(v_exp, v_exp, 'k--', linewidth=1, alpha=0.3, label='$V_{rad} = V_{exp}$')
ax.set_xlabel('$V_{exp}$ (km/s)')
ax.set_ylabel('$V_{rad}$ (km/s)')
ax.set_title(
    '$V_{rad}$ vs $V_{exp}$ (Schwenn 2005) / 방사 속도 vs 팽창 속도',
    fontweight='bold'
)
ax.legend()
ax.set_aspect('equal')
ax.set_xlim(0, 2500)
ax.set_ylim(0, 2500)

fig.suptitle(
    'CME Properties and Statistics / CME 특성과 통계',
    fontsize=14, fontweight='bold'
)
plt.tight_layout()
plt.show()

In [ ]:
# === CME Occurrence Rate vs Solar Cycle ===

# Synthetic solar cycle with ~11-year period.
years = np.linspace(1996, 2006, 120)  # SOHO era.
# Approximate solar cycle 23.
phase = 2 * np.pi * (years - 1996) / 11.0
ssn = 120 * np.sin(phase) ** 2 * np.where(phase < np.pi, 1.0, 0.6)
ssn += np.random.normal(0, 15, len(years))
ssn = np.clip(ssn, 0, 200)

# CME rate correlates with SSN but not linearly.
cme_rate = 0.5 + 5 * (ssn / 120) ** 0.8 + np.random.normal(0, 0.5, len(years))
cme_rate = np.clip(cme_rate, 0.3, 8)

fig, ax1 = plt.subplots(figsize=(12, 5))

color1 = '#1976D2'
ax1.bar(years, ssn, width=0.08, color=color1, alpha=0.6, label='SSN')
ax1.set_xlabel('Year / 연도')
ax1.set_ylabel('Sunspot Number', color=color1)
ax1.tick_params(axis='y', labelcolor=color1)

ax2 = ax1.twinx()
color2 = '#D32F2F'
ax2.plot(years, cme_rate, 'o-', color=color2, linewidth=2,
         markersize=4, label='CME rate')
ax2.set_ylabel('CME Rate (per day)', color=color2)
ax2.tick_params(axis='y', labelcolor=color2)

fig.suptitle(
    'CME Occurrence Rate vs Solar Cycle Phase\n'
    'CME 발생률 vs 태양 활동 주기',
    fontsize=14, fontweight='bold'
)

# Combined legend.
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

plt.tight_layout()
plt.show()

print("Solar minimum: ~0.5 CME/day")
print("Solar maximum: ~5-6 CME/day")
print("태양 활동 극소기: ~0.5 CME/일")
print("태양 활동 극대기: ~5-6 CME/일")

---
## Part 7: ICME 도착 시간 예측 / ICME Travel Time Prediction

Schwenn (2006)의 핵심 실용적 기여 중 하나는 CME 팽창 속도로부터
ICME 도착 시간을 예측하는 경험식이다 (Figure 35):

$$T_{tr} = 203 - 20.77 \times \ln(V_{exp}) \quad \text{[hours]}$$

이 공식은 표준 편차 14시간으로, ~1일 이내의 예보 정확도를 제공한다.
Drag-based model은 CME가 주변 태양풍과의 상호작용으로 감속/가속되는 과정을 기술한다.

One of Schwenn's key practical contributions is an empirical formula predicting
ICME travel time from CME expansion speed (Figure 35):

$$T_{tr} = 203 - 20.77 \times \ln(V_{exp}) \quad \text{[hours]}$$

This formula has a standard deviation of 14 hours, providing ~1-day forecast accuracy.
The drag-based model describes CME deceleration/acceleration via interaction
with the ambient solar wind.

In [ ]:
# === ICME Travel Time Prediction (Figure 35) ===

def schwenn_travel_time(v_exp):
    """Schwenn (2006) empirical travel time formula.

    T_tr = 203 - 20.77 * ln(V_exp) [hours]

    Args:
        v_exp: CME expansion speed (km/s).

    Returns:
        Travel time (hours).
    """
    return 203 - 20.77 * np.log(v_exp)


def constant_speed_travel_time(v):
    """Simple kinematic travel time at constant speed.

    T = 1 AU / V

    Args:
        v: Speed (km/s).

    Returns:
        Travel time (hours).
    """
    return (AU / 1e3) / v / 3600  # AU in km / speed / seconds to hours.


fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Panel 1: Schwenn formula vs constant speed.
ax = axes[0]
v_range = np.linspace(100, 3000, 500)

t_schwenn = schwenn_travel_time(v_range)
t_const = constant_speed_travel_time(v_range)

sigma = 14  # Standard deviation in hours.

ax.plot(v_range, t_schwenn, 'r-', linewidth=2.5,
        label='Schwenn: $T_{tr} = 203 - 20.77 \\ln(V_{exp})$')
ax.fill_between(v_range, t_schwenn - sigma, t_schwenn + sigma,
                alpha=0.2, color='red', label=f'+/- {sigma}h (1 sigma)')
ax.fill_between(v_range, t_schwenn - 2 * sigma, t_schwenn + 2 * sigma,
                alpha=0.1, color='red', label=f'+/- {2*sigma}h (2 sigma, ~95%)')
ax.plot(v_range, t_const, 'b--', linewidth=2,
        label='Constant speed: $T = 1\\,AU / V$')

# Generate synthetic data points.
np.random.seed(123)
n_obs = 80
v_obs = np.random.uniform(200, 2500, n_obs)
t_obs = schwenn_travel_time(v_obs) + np.random.normal(0, sigma, n_obs)
ax.scatter(v_obs, t_obs, s=20, c='gray', alpha=0.5, edgecolors='none',
           label='Synthetic observations')

ax.set_xlabel('CME Expansion Speed $V_{exp}$ (km/s)')
ax.set_ylabel('Travel Time (hours)')
ax.set_title(
    'ICME Travel Time vs CME Speed (Figure 35)\n'
    'ICME 도착 시간 vs CME 속도',
    fontweight='bold'
)
ax.set_ylim(0, 150)
ax.set_xlim(100, 3000)
ax.legend(fontsize=9, loc='upper right')

# Panel 2: Drag-based model.
ax = axes[1]


def drag_model(state, t_sec, gamma, v_sw):
    """Drag-based CME propagation model.

    dv/dt = -gamma * (v - v_sw) * |v - v_sw|
    dr/dt = v

    Args:
        state: [r (m), v (m/s)].
        t_sec: Time (seconds).
        gamma: Drag parameter (1/m).
        v_sw: Ambient solar wind speed (m/s).

    Returns:
        [dr/dt, dv/dt].
    """
    r, v = state
    dvdt = -gamma * (v - v_sw) * abs(v - v_sw)
    return [v, dvdt]


t_sim = np.linspace(0, 5 * 86400, 10000)  # 5 days in seconds.
gamma = 1e-7 / AU  # Typical drag parameter.

# Different initial CME speeds.
v0_list = [2000e3, 1000e3, 500e3, 300e3]  # m/s
v_sw_ambient = 400e3  # m/s

for v0 in v0_list:
    sol = odeint(drag_model, [20 * R_SUN, v0], t_sim,
                args=(gamma, v_sw_ambient))
    r_au = sol[:, 0] / AU
    v_kms = sol[:, 1] / 1e3

    # Find 1 AU crossing time.
    idx_1au = np.searchsorted(r_au, 1.0)
    if idx_1au < len(t_sim):
        t_1au_hr = t_sim[idx_1au] / 3600
        label = f'$V_0$ = {v0/1e3:.0f} km/s -> {t_1au_hr:.0f} h'
    else:
        label = f'$V_0$ = {v0/1e3:.0f} km/s'

    ax.plot(r_au, v_kms, linewidth=2, label=label)

ax.axhline(v_sw_ambient / 1e3, color='gray', linestyle=':', linewidth=1.5,
           label=f'$V_{{sw}}$ = {v_sw_ambient/1e3:.0f} km/s')
ax.axvline(1.0, color='green', linestyle='--', alpha=0.5, label='1 AU')

ax.set_xlabel('Distance (AU) / 거리')
ax.set_ylabel('CME Speed (km/s)')
ax.set_title(
    'Drag-Based Model: CME Speed Evolution\n'
    'Drag 모델: CME 속도 진화',
    fontweight='bold'
)
ax.set_xlim(0, 1.5)
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# === Effect of Ambient Solar Wind Speed on Travel Time ===

fig, ax = plt.subplots(figsize=(10, 6))

v0_cme = 1000e3  # Fixed initial CME speed: 1000 km/s.
v_sw_list = [300e3, 400e3, 500e3, 600e3]  # Different ambient wind speeds.
colors_sw = ['#1976D2', '#388E3C', '#F57C00', '#D32F2F']
t_sim = np.linspace(0, 5 * 86400, 10000)

for v_sw, color in zip(v_sw_list, colors_sw):
    sol = odeint(drag_model, [20 * R_SUN, v0_cme], t_sim,
                args=(gamma, v_sw))
    r_au = sol[:, 0] / AU
    t_hr = t_sim / 3600

    idx_1au = np.searchsorted(r_au, 1.0)
    if idx_1au < len(t_sim):
        t_arrival = t_hr[idx_1au]
        label = f'$V_{{sw}}$ = {v_sw/1e3:.0f} km/s -> arrival: {t_arrival:.0f} h'
        ax.plot(t_hr[:idx_1au + 100], r_au[:idx_1au + 100],
                color=color, linewidth=2, label=label)
        ax.scatter([t_arrival], [1.0], color=color, s=100, zorder=5,
                   edgecolor='black')
    else:
        ax.plot(t_hr, r_au, color=color, linewidth=2,
                label=f'$V_{{sw}}$ = {v_sw/1e3:.0f} km/s')

ax.axhline(1.0, color='green', linestyle='--', alpha=0.5)
ax.text(2, 1.02, 'Earth (1 AU)', color='green', fontsize=10)

ax.set_xlabel('Time (hours) / 시간')
ax.set_ylabel('Distance (AU) / 거리')
ax.set_title(
    f'Effect of Ambient Wind Speed on ICME Travel Time ($V_0$ = {v0_cme/1e3:.0f} km/s)\n'
    f'주변 태양풍 속도가 ICME 도착 시간에 미치는 영향',
    fontweight='bold'
)
ax.legend(fontsize=10)
ax.set_xlim(0, 100)
ax.set_ylim(0, 1.3)

plt.tight_layout()
plt.show()

---
## Part 8: 우주 기상 영향 체인 -- 태양에서 지구까지 / Space Weather Impact Chain -- From Sun to Earth

우주 기상 이벤트의 완전한 타임라인은 세 가지 주요 도착을 포함한다:
1. **전자기 복사 (빛)**: ~8분 (플레어 EUV, X선)
2. **태양 고에너지 입자 (SEP)**: ~30분 ~ 수 시간 (양성자, 전자)
3. **ICME**: ~1-4일 (플라즈마 + 자기장 구조)

ICME의 $B_z$ 남향 성분이 지자기 폭풍의 핵심 트리거이며,
Burton 공식으로 $D_{st}$ 지수의 시간 변화를 예측할 수 있다:

$$\frac{dD_{st}}{dt} = Q(t) - \frac{D_{st}}{\tau}$$

The complete timeline of a space weather event includes three major arrivals:
1. **EM radiation (light)**: ~8 min (flare EUV, X-rays)
2. **SEPs**: ~30 min to hours (protons, electrons)
3. **ICME**: ~1-4 days (plasma + magnetic structure)

The southward $B_z$ component of the ICME is the key trigger for geomagnetic storms,
and the Burton formula predicts the time evolution of the $D_{st}$ index.

In [ ]:
# === Space Weather Timeline ===

def compute_timeline(cme_speed_kms):
    """Compute arrival times for different space weather components.

    Args:
        cme_speed_kms: CME speed at the Sun (km/s).

    Returns:
        Dictionary of arrival times in hours.
    """
    # Light travel time: 1 AU / c.
    t_light = AU / constants.c / 60  # minutes

    # SEP arrival: protons at ~0.1c to ~0.5c along Parker spiral.
    # Spiral path length ~1.2 AU for well-connected events.
    spiral_length = 1.2 * AU
    v_sep_fast = 0.3 * constants.c  # Relativistic protons.
    v_sep_slow = 0.01 * constants.c  # ~10 MeV protons.
    t_sep_fast = spiral_length / v_sep_fast / 3600  # Hours.
    t_sep_slow = spiral_length / v_sep_slow / 3600  # Hours.

    # ICME arrival from Schwenn formula.
    t_icme = schwenn_travel_time(cme_speed_kms)

    return {
        'Light (X-ray/EUV)': t_light / 60,  # Convert to hours.
        'SEP (fast, ~0.3c)': t_sep_fast,
        'SEP (slow, ~0.01c)': t_sep_slow,
        'ICME': t_icme,
    }


# Compute for a fast CME (1500 km/s).
cme_v = 1500
timeline = compute_timeline(cme_v)

fig, ax = plt.subplots(figsize=(14, 5))

colors_tl = ['#FFC107', '#FF9800', '#FF5722', '#D32F2F']
y_pos = [3, 2, 1, 0]

for (name, t_hr), color, y in zip(timeline.items(), colors_tl, y_pos):
    ax.barh(y, t_hr, height=0.5, color=color, edgecolor='black',
            linewidth=1, alpha=0.8)
    if t_hr < 1:
        time_str = f'{t_hr * 60:.1f} min'
    else:
        time_str = f'{t_hr:.1f} h ({t_hr/24:.1f} d)'
    ax.text(t_hr + 1, y, time_str, va='center', fontsize=11, fontweight='bold')

ax.set_yticks(y_pos)
ax.set_yticklabels(list(timeline.keys()), fontsize=11)
ax.set_xlabel('Time after eruption (hours) / 분출 후 시간')
ax.set_title(
    f'Space Weather Timeline (CME speed = {cme_v} km/s)\n'
    f'우주 기상 타임라인 (CME 속도 = {cme_v} km/s)',
    fontsize=14, fontweight='bold'
)
ax.set_xlim(0, max(timeline.values()) * 1.3)

plt.tight_layout()
plt.show()

print("\n=== Space Weather Arrival Times / 우주 기상 도착 시간 ===")
for name, t_hr in timeline.items():
    if t_hr < 1:
        print(f"  {name}: {t_hr * 60:.1f} minutes")
    else:
        print(f"  {name}: {t_hr:.1f} hours ({t_hr/24:.1f} days)")

In [ ]:
# === Burton Formula: Geomagnetic Storm from ICME ===

def burton_dst(t_hours, v_sw_kms, n_p, bz_nt, dst0=0):
    """Simulate Dst evolution using the Burton et al. (1975) formula.

    dDst/dt = Q(t) - Dst/tau

    where Q depends on the solar wind electric field E_y = V_sw * B_s.

    Args:
        t_hours: Time array (hours).
        v_sw_kms: Solar wind speed time series (km/s).
        n_p: Proton density time series (cm^-3).
        bz_nt: IMF Bz time series (nT, negative = southward).
        dst0: Initial Dst value (nT).

    Returns:
        Dst time series (nT).
    """
    dt_hr = t_hours[1] - t_hours[0]
    dst = np.zeros_like(t_hours)
    dst[0] = dst0

    tau = 7.7  # Ring current decay time (hours).

    for i in range(1, len(t_hours)):
        # Southward Bz only (rectified).
        bs = max(0, -bz_nt[i])  # nT, positive when Bz is southward.

        # Dawn-dusk electric field: E_y = V_sw * B_s (mV/m).
        ey = v_sw_kms[i] * bs * 1e-3  # mV/m

        # Energy injection function Q (Burton formulation).
        # Q = -a * (E_y - 0.5) when E_y > 0.5 mV/m, else 0.
        a = 4.4  # nT/hr per mV/m, empirical constant.
        if ey > 0.5:
            Q = -a * (ey - 0.5)
        else:
            Q = 0

        # Pressure correction term.
        p_dyn = 1.67e-6 * n_p[i] * v_sw_kms[i] ** 2  # nPa
        b_corr = 7.26 * np.sqrt(p_dyn)  # Pressure correction.

        # Euler integration.
        ddst_dt = Q - (dst[i - 1] - b_corr) / tau
        dst[i] = dst[i - 1] + ddst_dt * dt_hr

    return dst


# === Simulate a synthetic ICME passage ===

t_hours = np.linspace(0, 96, 2000)  # 4 days.

# Background solar wind.
v_sw = np.full_like(t_hours, 400.0)  # km/s
n_p = np.full_like(t_hours, 5.0)  # cm^-3
bz = np.zeros_like(t_hours)  # nT

# ICME arrives at t = 24 hours.
t_icme_start = 24  # Shock arrival.
t_sheath_end = 30  # Sheath duration ~6 hours.
t_mc_start = 30  # Magnetic cloud start.
t_mc_end = 54  # Magnetic cloud end (~24 hours duration).

for i, t in enumerate(t_hours):
    if t_icme_start <= t < t_sheath_end:
        # Sheath: compressed, turbulent.
        v_sw[i] = 600
        n_p[i] = 20
        bz[i] = -5 + 10 * np.sin(2 * np.pi * (t - t_icme_start) / 3)
    elif t_mc_start <= t < t_mc_end:
        # Magnetic cloud: smooth rotation of B.
        phase = np.pi * (t - t_mc_start) / (t_mc_end - t_mc_start)
        v_sw[i] = 550 - 100 * (t - t_mc_start) / (t_mc_end - t_mc_start)
        n_p[i] = 8
        # Smooth Bz rotation: south -> north (or vice versa).
        bz[i] = -20 * np.cos(phase)  # Peak southward at the center.
    elif t >= t_mc_end:
        # Recovery.
        v_sw[i] = 450 - 50 * (1 - np.exp(-(t - t_mc_end) / 20))
        n_p[i] = 5

# Compute Dst.
dst = burton_dst(t_hours, v_sw, n_p, bz)

# === Plot ===
fig, axes = plt.subplots(4, 1, figsize=(14, 14), sharex=True)

# Solar wind speed.
axes[0].plot(t_hours, v_sw, 'k-', linewidth=1.5)
axes[0].set_ylabel('$V_{sw}$ (km/s)')
axes[0].set_title(
    'Synthetic ICME Passage and Geomagnetic Storm\n'
    '합성 ICME 통과와 지자기 폭풍',
    fontsize=14, fontweight='bold'
)
axes[0].axvspan(t_icme_start, t_sheath_end, alpha=0.2, color='orange',
               label='Sheath')
axes[0].axvspan(t_mc_start, t_mc_end, alpha=0.2, color='blue',
               label='Magnetic cloud')
axes[0].legend(loc='upper right')

# Proton density.
axes[1].plot(t_hours, n_p, 'g-', linewidth=1.5)
axes[1].set_ylabel('$n_p$ (cm$^{-3}$)')
axes[1].axvspan(t_icme_start, t_sheath_end, alpha=0.2, color='orange')
axes[1].axvspan(t_mc_start, t_mc_end, alpha=0.2, color='blue')

# IMF Bz.
axes[2].plot(t_hours, bz, 'b-', linewidth=1.5)
axes[2].fill_between(t_hours, 0, bz, where=bz < 0, alpha=0.3, color='red',
                     label='$B_z$ south (geoeffective)')
axes[2].axhline(0, color='k', linewidth=0.5)
axes[2].set_ylabel('$B_z$ (nT)')
axes[2].axvspan(t_icme_start, t_sheath_end, alpha=0.2, color='orange')
axes[2].axvspan(t_mc_start, t_mc_end, alpha=0.2, color='blue')
axes[2].legend(loc='lower right')

# Dst index.
axes[3].plot(t_hours, dst, 'r-', linewidth=2.5)
axes[3].axhline(0, color='k', linewidth=0.5)
axes[3].axhline(-50, color='orange', linestyle='--', alpha=0.5, label='Moderate storm')
axes[3].axhline(-100, color='red', linestyle='--', alpha=0.5, label='Intense storm')
axes[3].set_ylabel('$D_{st}$ (nT)')
axes[3].set_xlabel('Time (hours) / 시간')
axes[3].axvspan(t_icme_start, t_sheath_end, alpha=0.2, color='orange')
axes[3].axvspan(t_mc_start, t_mc_end, alpha=0.2, color='blue')
axes[3].legend(loc='lower right')

min_dst = np.min(dst)
min_dst_t = t_hours[np.argmin(dst)]
axes[3].annotate(
    f'Min $D_{{st}}$ = {min_dst:.0f} nT\nat t = {min_dst_t:.0f} h',
    xy=(min_dst_t, min_dst),
    xytext=(min_dst_t + 15, min_dst + 30),
    fontsize=11, fontweight='bold',
    arrowprops=dict(arrowstyle='->', color='red'),
    color='red'
)

plt.tight_layout()
plt.show()

# Summary.
print("=" * 60)
print("Geomagnetic Storm Summary / 지자기 폭풍 요약")
print("=" * 60)
print(f"ICME shock arrival: t = {t_icme_start} h")
print(f"Minimum Dst: {min_dst:.0f} nT at t = {min_dst_t:.0f} h")
if min_dst > -50:
    category = "Weak / 약한 폭풍"
elif min_dst > -100:
    category = "Moderate / 보통 폭풍"
elif min_dst > -250:
    category = "Intense / 강한 폭풍"
else:
    category = "Super-intense / 초강력 폭풍"
print(f"Storm category: {category}")

---
## 핵심 요약 / Key Takeaways

1. **태양풍 두 상태 / Two-state wind**: 고속풍(700 km/s)과 저속풍(350 km/s)은 속도와 밀도가 크게 다르지만, 운동량/에너지 플럭스 밀도는 놀랍도록 유사하다.
   HSS (700 km/s) and LSM (350 km/s) differ greatly in speed and density, yet momentum/energy flux densities are remarkably similar.

2. **Parker spiral**: 느린 바람은 더 단단히 감기고, 빠른 바람이 느린 바람을 따라잡으면 CIR이 형성된다.
   Slow wind is wound more tightly; CIRs form where fast wind overtakes slow wind.

3. **발레리나 모델 / Ballerina model**: HCS의 3D 구조는 태양 쌍극자 기울기에 따라 변하며, 섹터 경계에서 $B_z$ 변동이 발생한다.
   The 3D HCS structure varies with dipole tilt, producing $B_z$ variations at sector boundaries.

4. **라디오 버스트 / Radio bursts**: Type III (빠른 전자 빔)와 Type II (충격파)는 코로나 밀도 모델을 통해 주파수-거리 관계로 추적 가능하다.
   Type III (fast electron beams) and Type II (shocks) can be tracked via frequency-distance relations using coronal density models.

5. **SEP 프로파일 / SEP profiles**: 관측자의 자기적 연결 위치에 따라 시간-강도 프로파일이 극적으로 달라진다.
   Time-intensity profiles vary dramatically with observer's magnetic connection.

6. **CME 통계 / CME statistics**: $V_{rad} = 0.88 V_{exp}$ 관계로 투영 효과를 극복할 수 있다.
   The $V_{rad} = 0.88 V_{exp}$ relation overcomes projection effects.

7. **도착 시간 예측 / Travel time**: Schwenn 경험식($T_{tr} = 203 - 20.77 \ln V_{exp}$)은 +/-14시간 정확도를 제공하며, drag 모델은 물리적 이해를 더한다.
   Schwenn's formula provides +/-14h accuracy; the drag model adds physical understanding.

8. **Burton 공식 / Burton formula**: ICME의 $B_z$ 남향 성분이 $D_{st}$ 감소(지자기 폭풍)의 핵심 드라이버이며, 고리 전류의 주입-감쇠 균형으로 기술된다.
   Southward $B_z$ drives $D_{st}$ decrease (geomagnetic storms), described by injection-decay balance of the ring current.

---

## 참고 문헌 / References

- Schwenn, R., "Space Weather: The Solar Perspective", *Living Rev. Solar Phys.*, 3, 2 (2006). DOI: 10.12942/lrsp-2006-2
- Parker, E.N., "Dynamics of the Interplanetary Gas and Magnetic Fields", *ApJ*, 128, 664 (1958)
- Burton, R.K., McPherron, R.L., Russell, C.T., "An empirical relationship between interplanetary conditions and Dst", *JGR*, 80, 4204 (1975)
- Newkirk, G., "The Solar Corona in Active Regions and the Thermal Origin of the Slowly Varying Component of Solar Radio Radiation", *ApJ*, 133, 983 (1961)
- Alfven, H., "Electric currents in cosmic plasmas", *Rev. Geophys.*, 15, 271 (1977)